In [0]:
from pyspark.sql import functions as F

sales = spark.table("retailnew.gold.fact_sales")

clv = (
    sales.groupBy("customer_id")
    .agg(
        F.sum("sales_amount").alias("total_spent"),
        F.count("transaction_id").cast("int").alias("frequency"),
        F.max("transaction_date").alias("last_purchase")
    )
    .withColumn("recency_days", F.datediff(F.current_date(), F.col("last_purchase")))
    .withColumn(
        "clv_score",
        F.expr("0.5 * CAST(frequency AS int) + 0.3 * total_spent - 0.2 * recency_days")
    )
)

clv.show()

(
    clv.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("retailnew.gold.customer_clv")
)
